# 🌦️ Pakistan Weather Analysis 2024–2025
### Exploratory Data Analysis + Machine Learning
**Author:** Hassan Ali | [Kaggle: hassanali789](https://www.kaggle.com/hassanali789)

This notebook provides a complete analysis of daily weather data for 10 major Pakistani cities over 2 years.
We explore temperature patterns, rainfall, humidity, extreme weather events, and build a temperature forecasting model.

---


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette("husl")

print("All libraries loaded successfully!")

## 2. Load & Inspect Dataset

In [ ]:
df = pd.read_csv("/kaggle/input/pakistan-weather-2024-2025/pakistan_weather_2024_2025.csv",
                 parse_dates=["date"])

print(f"Shape        : {df.shape}")
print(f"Date range   : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Cities       : {sorted(df['city'].unique())}")
print(f"Missing values:\n{df.isnull().sum()}")
df.head(10)

## 3. Statistical Summary

In [ ]:
print("=== Data Types ===")
print(df.dtypes)
print()
print("=== Descriptive Statistics ===")
df.describe().round(2)

## 4. Average Temperature by City

In [ ]:
city_stats = df.groupby("city").agg(
    mean_temp  = ("temp_mean_c",  "mean"),
    max_temp   = ("temp_max_c",   "max"),
    min_temp   = ("temp_min_c",   "min"),
    total_rain = ("precipitation_mm", "sum")
).round(2).sort_values("mean_temp", ascending=False).reset_index()

fig, ax = plt.subplots()
colors = ["#d62728" if t > 27 else "#1f77b4" for t in city_stats["mean_temp"]]
bars = ax.bar(city_stats["city"], city_stats["mean_temp"], color=colors, edgecolor="white", linewidth=0.5)
ax.bar_label(bars, fmt="%.1f°C", padding=3, fontsize=9)
ax.set_title("Average Annual Temperature by City", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Mean Temperature (°C)")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("avg_temp_by_city.png", dpi=150, bbox_inches="tight")
plt.show()

print(city_stats.to_string(index=False))

## 5. Monthly Temperature Trends (All Cities)

In [ ]:
df["month"] = df["date"].dt.month
df["year"]  = df["date"].dt.year
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

monthly = df.groupby(["city","month"])["temp_mean_c"].mean().reset_index()

fig, ax = plt.subplots(figsize=(13, 5))
for city in sorted(df["city"].unique()):
    d = monthly[monthly["city"] == city]
    ax.plot(d["month"], d["temp_mean_c"], marker="o", markersize=4, label=city, linewidth=1.8)

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.set_title("Monthly Mean Temperature by City", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Temperature (°C)")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig("monthly_temp_trends.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Temperature Heatmap — City vs Month

In [ ]:
pivot = df.pivot_table(values="temp_mean_c", index="city", columns="month", aggfunc="mean")
pivot.columns = month_names

fig, ax = plt.subplots(figsize=(13, 6))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlBu_r",
            linewidths=0.4, linecolor="white",
            cbar_kws={"label": "Mean Temp (°C)"}, ax=ax)
ax.set_title("Monthly Mean Temperature Heatmap (°C)", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("temp_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Rainfall Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Total annual rainfall by city
city_rain = df.groupby("city")["precipitation_mm"].sum().sort_values(ascending=False)
axes[0].bar(city_rain.index, city_rain.values, color="#2196F3", edgecolor="white", linewidth=0.5)
axes[0].set_title("Total Rainfall by City (2024–2025)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Total Precipitation (mm)")
axes[0].tick_params(axis="x", rotation=30)

# Monthly average rainfall
monthly_rain = df.groupby("month")["precipitation_mm"].mean()
axes[1].bar(month_names, monthly_rain.values, color="#64B5F6", edgecolor="white", linewidth=0.5)
axes[1].set_title("Average Monthly Rainfall (All Cities)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Avg Precipitation (mm)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("rainfall_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Extreme Heat Events — Days Above 45°C

In [ ]:
extreme = df[df["temp_max_c"] >= 45].copy()
extreme_count = extreme.groupby("city").size().sort_values(ascending=False).reset_index()
extreme_count.columns = ["city", "days_above_45c"]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(extreme_count["city"], extreme_count["days_above_45c"],
              color="#E53935", edgecolor="white", linewidth=0.5)
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_title("Number of Days with Temperature ≥ 45°C", fontsize=14, fontweight="bold", pad=12)
ax.set_ylabel("Days")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("extreme_heat.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nTop extreme heat days:")
print(extreme[["city","date","temp_max_c"]].sort_values("temp_max_c", ascending=False).head(10).to_string(index=False))

## 9. Correlation Between Weather Variables

In [ ]:
numeric_cols = ["temp_max_c","temp_min_c","temp_mean_c",
                "precipitation_mm","wind_max_kmh","humidity_pct"]

corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, linewidths=0.4,
            linecolor="white", ax=ax)
ax.set_title("Correlation Matrix — Weather Variables", fontsize=14, fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig("correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Jacobabad Deep Dive 🌡️
> Jacobabad is considered one of the **hottest inhabited cities on Earth**, regularly exceeding 50°C in summer.

In [ ]:
jb = df[df["city"] == "Jacobabad"].copy().sort_values("date")

fig, axes = plt.subplots(2, 1, figsize=(13, 8))

# Daily temp over time
axes[0].fill_between(jb["date"], jb["temp_min_c"], jb["temp_max_c"], alpha=0.3, color="#E53935")
axes[0].plot(jb["date"], jb["temp_mean_c"], color="#B71C1C", linewidth=1.2, label="Mean Temp")
axes[0].axhline(50, color="black", linestyle="--", linewidth=0.8, label="50°C threshold")
axes[0].set_title("Jacobabad Daily Temperature Range", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Temperature (°C)")
axes[0].legend()

# Rainfall
axes[1].bar(jb["date"], jb["precipitation_mm"], color="#1565C0", width=1.5)
axes[1].set_title("Jacobabad Daily Rainfall", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Precipitation (mm)")

plt.tight_layout()
plt.savefig("jacobabad_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Max temperature ever recorded : {jb['temp_max_c'].max()}°C on {jb.loc[jb['temp_max_c'].idxmax(), 'date'].date()}")
print(f"Days above 50°C               : {(jb['temp_max_c'] >= 50).sum()}")
print(f"Total annual rainfall         : {jb['precipitation_mm'].sum():.1f} mm")

## 11. Feature Engineering for ML

In [ ]:
df_ml = df.copy()

le = LabelEncoder()
df_ml["city_enc"] = le.fit_transform(df_ml["city"])

df_ml["day_of_year"] = df_ml["date"].dt.dayofyear
df_ml["week"]        = df_ml["date"].dt.isocalendar().week.astype(int)
df_ml["month_num"]   = df_ml["date"].dt.month
df_ml["season"]      = df_ml["month_num"].map({
    12:0, 1:0, 2:0,   # Winter
     3:1, 4:1, 5:1,   # Spring
     6:2, 7:2, 8:2,   # Summer
     9:3,10:3,11:3    # Autumn
})

df_ml = df_ml.sort_values(["city","date"])
df_ml["temp_lag1"] = df_ml.groupby("city")["temp_mean_c"].shift(1)
df_ml["temp_lag7"] = df_ml.groupby("city")["temp_mean_c"].shift(7)
df_ml["temp_roll7"] = df_ml.groupby("city")["temp_mean_c"].transform(lambda x: x.rolling(7).mean())

df_ml = df_ml.dropna()
print(f"ML-ready shape: {df_ml.shape}")
print(f"Features created: day_of_year, week, month_num, season, city_enc, temp_lag1, temp_lag7, temp_roll7")
df_ml.head()

## 12. Model Training — Predicting Next Day Temperature

In [ ]:
features = ["city_enc","day_of_year","week","month_num","season",
            "temp_lag1","temp_lag7","temp_roll7",
            "humidity_pct","wind_max_kmh","precipitation_mm"]
target = "temp_mean_c"

X = df_ml[features]
y = df_ml[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train size: {X_train.shape[0]:,} | Test size: {X_test.shape[0]:,}")

models = {
    "Linear Regression"        : LinearRegression(),
    "Random Forest"            : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting"        : GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    r2   = r2_score(y_test, preds)
    results.append({"Model": name, "MAE": round(mae,3), "RMSE": round(rmse,3), "R²": round(r2,4)})
    trained_models[name] = (model, preds)
    print(f"{name:28s} → MAE: {mae:.3f}°C | RMSE: {rmse:.3f}°C | R²: {r2:.4f}")

results_df = pd.DataFrame(results).sort_values("R²", ascending=False)
print("\nBest model:", results_df.iloc[0]["Model"])

## 13. Model Comparison & Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model comparison bar chart
metrics = ["MAE", "RMSE"]
x = np.arange(len(results_df))
width = 0.35
axes[0].bar(x - width/2, results_df["MAE"],  width, label="MAE",  color="#42A5F5")
axes[0].bar(x + width/2, results_df["RMSE"], width, label="RMSE", color="#EF5350")
axes[0].set_xticks(x)
axes[0].set_xticklabels(results_df["Model"], rotation=15, ha="right")
axes[0].set_title("Model Comparison — MAE & RMSE", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Error (°C)")
axes[0].legend()

# Feature importance (Random Forest)
rf_model = trained_models["Random Forest"][0]
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)
axes[1].barh(importances.index, importances.values, color="#66BB6A")
axes[1].set_title("Feature Importance — Random Forest", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Importance Score")

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 14. Actual vs Predicted — Best Model

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_preds = trained_models[best_name][1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
axes[0].scatter(y_test, best_preds, alpha=0.3, s=10, color="#1565C0")
lims = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
axes[0].plot(lims, lims, "r--", linewidth=1.2, label="Perfect prediction")
axes[0].set_xlabel("Actual Temperature (°C)")
axes[0].set_ylabel("Predicted Temperature (°C)")
axes[0].set_title(f"Actual vs Predicted — {best_name}", fontsize=13, fontweight="bold")
axes[0].legend()

# Residuals
residuals = y_test.values - best_preds
axes[1].hist(residuals, bins=40, color="#42A5F5", edgecolor="white", linewidth=0.3)
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.2)
axes[1].set_title("Residual Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Residual (°C)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Best model : {best_name}")
print(f"R² Score   : {results_df.iloc[0]['R²']}")
print(f"MAE        : {results_df.iloc[0]['MAE']}°C")

## 15. Key Findings & Conclusions

### Temperature
- **Jacobabad** is the hottest city with the highest number of days exceeding 45°C and 50°C
- **Quetta** and **Islamabad** have the most moderate temperatures due to elevation
- Peak summer temperatures occur in **June–July** across all cities

### Rainfall
- **Lahore** and **Islamabad** receive the most rainfall, primarily during the monsoon season (July–September)
- **Quetta** and **Karachi** are the driest cities
- Monsoon season accounts for the majority of annual precipitation

### Machine Learning
- **Random Forest** and **Gradient Boosting** outperform Linear Regression significantly
- Temperature lag features (previous day, previous week) are the most important predictors
- The model achieves strong R² scores, confirming weather patterns are highly predictable short-term

---

*Dataset by Hassan Ali | hassanali789 on Kaggle*  
*Source: Open-Meteo Historical Archive*
